为了方便演示，我们假设有一个形状为 (4, 8) 的张量 data（代表 4 个样本，每个样本 8 个特征）：

In [1]:
import torch

# 模拟数据：4个样本，8个特征
data = torch.arange(32).reshape(4, 8)
print(f"原始数据 Shape: {data.shape}") 
# 输出: torch.Size([4, 8])

原始数据 Shape: torch.Size([4, 8])


# torch.split：处理固定大小的 Batch 数据

- 场景痛点：
  - 在模型推理或训练时，显存是有限的。如果你有一个包含 10,000 张图片的大张量，直接塞进 GPU 会爆显存。你需要把它切成固定大小的小块（**例如每块 64 张**），分批次处理。
- 为什么用 split：
  - <font color='red'>因为业务逻辑要求“每批次最多 N 个”，而不是“总共切 M 份”。</font>
    - 如果总数不能被 N 整除，最后一份会小于 N，split 能完美处理这种情况。

In [3]:
batch_size = 2 # 假设显存限制，每次只能处理2个样本

# 沿着第0维（样本维），每2个切一块
batches = torch.split(data,batch_size,dim=0)

print(f"切分成了{len(batches)}个批次")

# 模拟逐个批次送入GPU处理
for i ,batch in enumerate(batches):
    print(f"---处理第{i+1}批---")
    print(f"当前批次shape:{batch.shape}") # 始终是（2,8）
    

切分成了2个批次
---处理第1批---
当前批次shape:torch.Size([2, 8])
---处理第2批---
当前批次shape:torch.Size([2, 8])


## 💡 关键点：
- 即使你的数据有 5 个样本（batch_size=2），split 也会切出 (2,8), (2,8), (1,8) 三个块，保证逻辑正确。

# torch.chunk：将任务均分给多个 GPU/线程
- 场景痛点：
  - 你有 4 张显卡（GPU 0, 1, 2, 3），或者开启了 4 个 CPU 线程进行并行计算。<font color='red'>你希望把数据平均分配给它们，让它们同时开工，最后汇总结果。</font>
- 为什么用 chunk：
  - 因为资源数量是固定的（例如就是 4 卡）。**你不需要关心每块具体有多少数据**，<font color='red'>你只关心“我要切成 4 份”，让每一份对应一个计算单元。</font>

In [4]:
num_gpus = 4  # 假设我们有 4 张显卡

# 沿着第0维（样本维），平均切成 4 份
chunks = torch.chunk(data, chunks=num_gpus, dim=0)

print(f"任务已分配给 {len(chunks)} 个设备")

# 模拟并行分发（实际项目中通常配合 DataParallel 或 DDP）
for gpu_id, chunk_data in enumerate(chunks):
    print(f"GPU {gpu_id} 接收到的数据 Shape: {chunk_data.shape}")
    # device = torch.device(f'cuda:{gpu_id}')
    # chunk_data = chunk_data.to(device)
    # start_processing(chunk_data)

任务已分配给 4 个设备
GPU 0 接收到的数据 Shape: torch.Size([1, 8])
GPU 1 接收到的数据 Shape: torch.Size([1, 8])
GPU 2 接收到的数据 Shape: torch.Size([1, 8])
GPU 3 接收到的数据 Shape: torch.Size([1, 8])


## ⚠️ 注意：
- chunk 会尽力均分。如果数据是 10 个样本分给 3 个 GPU，它会切出 (4, 8), (3, 8), (3, 8)。第一块可能会多一个或少一个，但总数一定是你指定的份数。

## 📌 总结对比

| 函数 | 核心逻辑 | 典型应用 | 形象比喻 |
| :--- | :--- | :--- | :--- |
| `split` | 按大小切 (`size`) | 分批训练、滑动窗口、固定长度截断 | 切香肠：我要每片 2厘米 厚，不管能切几片。 |
| `chunk` | 按份数切 (`chunks`) | 多卡并行、多线程处理、K折交叉验证 | 分蛋糕：我们要 4个人 吃，尽量平分，不管每人分多少。 |



# “拆分（Split）→ 处理（Process）→ 拼接（Cat）”

为了让你直观看到数据形状的变化，我编写了一段完整的代码。这段代码模拟了以下场景：
1. 准备数据：
   - 生成一个包含 8 个样本的大张量。
2. 拆分 (Split)：
   - 因为显存限制，每次只能处理 3 个样本（Batch Size = 3）。
3. 模拟处理：
   - 对每个小批次进行计算（例如乘以 2）。
4. 拼接 (Cat)：
   - 将处理完的小块重新拼回完整的结果。

In [5]:
import torch

# ==========================================
# 1. 准备数据
# ==========================================
# 模拟一个包含 8 个样本，每个样本有 4 个特征的数据集
# Shape: (8, 4)
data = torch.arange(32).reshape(8, 4).float()
print(f"原始数据 Shape: {data.shape}")
print(f"原始数据内容:\n{data}\n")

# ==========================================
# 2. 拆分 (Split) - 对应场景：处理固定大小 Batch
# ==========================================
batch_size = 3  # 假设显存限制，一次只能处理 3 个样本

# 沿着第 0 维（样本维度）进行切分
# 注意：8 个样本除以 3，会得到 [3, 3, 2] 三个块
batches = torch.split(data, batch_size, dim=0)

print(f"拆分后共有 {len(batches)} 个批次")
for i, b in enumerate(batches):
    print(f"  - Batch {i+1} Shape: {b.shape}")

# ==========================================
# 3. 模拟处理与收集
# ==========================================
processed_batches = [] # 用于存储处理后的结果

print("\n开始逐批次处理...")
for i, batch in enumerate(batches):
    # --- 模拟 GPU 计算 ---
    # 假设模型逻辑是：所有数值乘以 2
    output = batch * 2 
    
    # 将结果存入列表
    processed_batches.append(output)
    print(f"  - Batch {i+1} 处理完成, Shape: {output.shape}")

# ==========================================
# 4. 拼接 (Cat) - 逆操作
# ==========================================
# 使用 torch.cat 将列表中的张量沿第 0 维重新拼合
final_result = torch.cat(processed_batches, dim=0)

print(f"\n最终结果 Shape: {final_result.shape}")
print(f"最终结果内容:\n{final_result}")

# ==========================================
# 5. 验证正确性
# ==========================================
expected = data * 2
if torch.equal(final_result, expected):
    print("\n✅ 验证通过：拆分-处理-拼接后的结果与直接计算一致！")
else:
    print("\n❌ 验证失败")

原始数据 Shape: torch.Size([8, 4])
原始数据内容:
tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.],
        [12., 13., 14., 15.],
        [16., 17., 18., 19.],
        [20., 21., 22., 23.],
        [24., 25., 26., 27.],
        [28., 29., 30., 31.]])

拆分后共有 3 个批次
  - Batch 1 Shape: torch.Size([3, 4])
  - Batch 2 Shape: torch.Size([3, 4])
  - Batch 3 Shape: torch.Size([2, 4])

开始逐批次处理...
  - Batch 1 处理完成, Shape: torch.Size([3, 4])
  - Batch 2 处理完成, Shape: torch.Size([3, 4])
  - Batch 3 处理完成, Shape: torch.Size([2, 4])

最终结果 Shape: torch.Size([8, 4])
最终结果内容:
tensor([[ 0.,  2.,  4.,  6.],
        [ 8., 10., 12., 14.],
        [16., 18., 20., 22.],
        [24., 26., 28., 30.],
        [32., 34., 36., 38.],
        [40., 42., 44., 46.],
        [48., 50., 52., 54.],
        [56., 58., 60., 62.]])

✅ 验证通过：拆分-处理-拼接后的结果与直接计算一致！
